# Data Modelling — Classification

*Generated by DoML — Do Machine Learning*

> **Note:** This notebook uses Python regardless of the `analysis_language` config setting.
> scikit-learn, SHAP, and Optuna do not have comparable R equivalents.

In [ ]:
# Reproducibility — REPR-01 (non-negotiable per CLAUDE.md)
import os, random
import numpy as np
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import json, warnings
from pathlib import Path
from datetime import date

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from sklearn.pipeline import Pipeline
import xgboost as xgb
import lightgbm as lgb
import shap
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings('ignore')

In [ ]:
# Project root — REPR-02 (non-negotiable per CLAUDE.md)
PROJECT_ROOT = Path(os.environ.get('PROJECT_ROOT', '/home/jovyan/work'))

with open(PROJECT_ROOT / '.planning' / 'config.json') as f:
    config = json.load(f)

problem_type = config.get('problem_type', 'classification')
TARGET = config.get('target_column')
row_count_threshold = 2000

valid_types = ('classification', 'binary_classification', 'multiclass_classification')
if problem_type not in valid_types:
    raise ValueError(f"This notebook is for classification problems. problem_type='{problem_type}' found in config.json.")

print(f"Project root: {PROJECT_ROOT}")
print(f"Problem type: {problem_type}")
print(f"Target column: {TARGET}")

In [ ]:
processed_dir = PROJECT_ROOT / 'data' / 'processed'
preprocessed_files = (list(processed_dir.glob('preprocessed_*.csv')) +
                      list(processed_dir.glob('preprocessed_*.parquet')))

if not preprocessed_files:
    raise FileNotFoundError(
        "No preprocessed dataset found. Run the preprocessing notebook first."
    )

input_file = preprocessed_files[0]
print(f"Loading: {input_file.name}")

if input_file.suffix == '.parquet':
    df = pd.read_parquet(input_file)
else:
    df = pd.read_csv(input_file)

if TARGET not in df.columns:
    raise ValueError(f"Target column '{TARGET}' not found. Check config.json target_column.")

X = df.drop(columns=[TARGET])
y = df[TARGET]

# Detect binary vs multi-class
n_classes = y.nunique()
is_binary = n_classes == 2

if is_binary:
    primary_metric = 'roc_auc'
    primary_label = 'ROC-AUC'
    cv_scoring = {'auc': 'roc_auc', 'f1': 'f1', 'accuracy': 'accuracy'}
    optuna_direction = 'maximize'
else:
    primary_metric = 'f1_macro'
    primary_label = 'Macro F1'
    cv_scoring = {'f1_macro': 'f1_macro', 'f1_weighted': 'f1_weighted', 'accuracy': 'accuracy'}
    optuna_direction = 'maximize'

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Target: {TARGET}  ({n_classes} classes)")
print(f"Type: {'binary' if is_binary else 'multi-class'}")
print(f"Primary metric: {primary_label}")

## Step 1: Holdout Split

80% training / 20% test with `stratify=y` to preserve class proportions. The test set is **sealed** until Step 5 (final model evaluation). No model selection decisions will be made using test set metrics — only CV metrics inform model selection.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"Train: {X_train.shape[0]:,} rows  |  Test: {X_test.shape[0]:,} rows")

# Recommend 10-fold CV for small datasets
n_folds = 5
if len(X_train) < row_count_threshold:
    print(f"\n⚠️  Small training set ({len(X_train):,} rows < {row_count_threshold}). "
          f"Consider using 10-fold CV by setting n_folds = 10 below.")
print(f"Using {n_folds}-fold StratifiedKFold CV.")

## Step 2: Model Definitions

Models evaluated:
- **DummyClassifier(most_frequent)** — baseline; every model must beat this (MOD-07)
- **LogisticRegression** — linear baseline
- **RandomForestClassifier** — ensemble, 100 trees
- **XGBClassifier** — gradient boosted trees (XGBoost)
- **LGBMClassifier** — gradient boosted trees (LightGBM)

In [ ]:
models = {
    'DummyClassifier(most_frequent)': DummyClassifier(strategy='most_frequent', random_state=SEED),
    'LogisticRegression':             LogisticRegression(max_iter=1000, random_state=SEED),
    'RandomForestClassifier':         RandomForestClassifier(n_estimators=100, random_state=SEED),
    'XGBClassifier':                  xgb.XGBClassifier(n_estimators=100, random_state=SEED,
                                                         eval_metric='logloss', verbosity=0),
    'LGBMClassifier':                 lgb.LGBMClassifier(n_estimators=100, random_state=SEED, verbose=-1),
}

## Step 3: Cross-Validated Leaderboard

StratifiedKFold CV on the training set. Metrics shown are mean ± std across folds.

**Primary metric (binary):** ROC-AUC (higher is better)
**Primary metric (multi-class):** Macro F1 (higher is better)

The test set is NOT used here — only CV scores inform model selection (MOD-06).

In [ ]:
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
leaderboard_rows = []

# Store fitted models and first-split data for SHAP
fitted_models = {}
shap_data_dict = {}

for name, model in models.items():
    scores = cross_validate(
        model, X_train, y_train, cv=skf,
        scoring=cv_scoring,
        return_estimator=True,
        return_train_score=False,
    )
    
    primary_key = 'test_auc' if is_binary else 'test_f1_macro'
    primary_mean = scores[primary_key].mean()
    primary_std  = scores[primary_key].std()
    
    if is_binary:
        f1_mean  = scores['test_f1'].mean()
        acc_mean = scores['test_accuracy'].mean()
    else:
        f1_mean  = scores['test_f1_weighted'].mean()
        acc_mean = scores['test_accuracy'].mean()
    
    leaderboard_rows.append({
        'model':              name,
        'cv_primary_mean':    round(primary_mean, 4),
        'cv_primary_std':     round(primary_std, 4),
        'cv_f1_mean':         round(f1_mean, 4),
        'cv_accuracy_mean':   round(acc_mean, 4),
        'test_score':         None,
        'note':               'CV',
    })
    
    # Save first-fold fitted model + validation split for SHAP
    first_split = list(skf.split(X_train, y_train))[0]
    _, val_idx = first_split
    X_val_shap = X_train.iloc[val_idx]
    fitted_models[name] = scores['estimator'][0]
    shap_data_dict[name] = X_val_shap
    
    print(f"{name:35s}  {primary_label}={primary_mean:.4f} ±{primary_std:.4f}  F1={f1_mean:.4f}  Acc={acc_mean:.4f}")

# Sort descending (higher is better for classification)
leaderboard = pd.DataFrame(leaderboard_rows).sort_values('cv_primary_mean', ascending=False).reset_index(drop=True)
print(f"\n--- LEADERBOARD (sorted by {primary_label}) ---")
display(leaderboard[['model','cv_primary_mean','cv_primary_std','cv_f1_mean','cv_accuracy_mean']])

In [ ]:
models_dir = PROJECT_ROOT / 'models'
models_dir.mkdir(exist_ok=True)
leaderboard_path = models_dir / 'leaderboard.csv'

if leaderboard_path.exists():
    existing = pd.read_csv(leaderboard_path)
    leaderboard_all = pd.concat([existing, leaderboard], ignore_index=True)
else:
    leaderboard_all = leaderboard.copy()

leaderboard_all.to_csv(leaderboard_path, index=False)
print(f"Leaderboard saved to {leaderboard_path.relative_to(PROJECT_ROOT)}")
print(f"Total rows in leaderboard: {len(leaderboard_all)}")

## Step 4: SHAP Explainability

SHAP values computed for every model on the **validation fold of the first CV split**.

For binary classification tree models, the positive class (index 1) shap values are used.
For multi-class, the mean across classes is used.

Plots saved to `reports/shap_{model_name}.png` (MOD-08).

In [ ]:
reports_dir = PROJECT_ROOT / 'reports'
reports_dir.mkdir(exist_ok=True)

TREE_MODELS = ('RandomForestClassifier', 'XGBClassifier', 'LGBMClassifier')
LINEAR_MODELS = ('LogisticRegression',)
SKIP_MODELS = ('DummyClassifier(most_frequent)',)

for name, fitted_model in fitted_models.items():
    if name in SKIP_MODELS:
        print(f"{name}: skipped (no meaningful SHAP explanation)")
        continue
    
    X_shap = shap_data_dict[name]
    
    try:
        if name in TREE_MODELS:
            explainer = shap.TreeExplainer(fitted_model)
            shap_vals = explainer.shap_values(X_shap)
            # Binary: shap_vals is list of 2 arrays — use positive class
            if isinstance(shap_vals, list) and len(shap_vals) == 2:
                shap_vals = shap_vals[1]
            elif isinstance(shap_vals, list):
                shap_vals = np.array(shap_vals).mean(axis=0)  # multi-class: mean across classes
        elif name in LINEAR_MODELS:
            explainer = shap.LinearExplainer(fitted_model, X_shap)
            shap_vals = explainer.shap_values(X_shap)
            if isinstance(shap_vals, list):
                shap_vals = shap_vals[1] if len(shap_vals) == 2 else np.array(shap_vals).mean(axis=0)
        else:
            background = shap.sample(X_shap, min(100, len(X_shap)))
            explainer = shap.KernelExplainer(fitted_model.predict_proba, background)
            shap_vals = explainer.shap_values(X_shap, nsamples=100)
            if isinstance(shap_vals, list):
                shap_vals = shap_vals[1] if len(shap_vals) == 2 else np.array(shap_vals).mean(axis=0)
        
        plt.figure(figsize=(8, 5))
        shap.summary_plot(shap_vals, X_shap, plot_type='bar', show=False,
                          max_display=10, title=f'SHAP — {name}')
        safe_name = name.replace('(', '').replace(')', '').replace(' ', '_')
        plot_path = reports_dir / f'shap_{safe_name}.png'
        plt.savefig(plot_path, bbox_inches='tight', dpi=100)
        plt.show()
        print(f"Saved: {plot_path.relative_to(PROJECT_ROOT)}")
    
    except Exception as e:
        print(f"{name}: SHAP failed — {e}")

## Step 5: Optuna Hyperparameter Tuning

Tuning the **top 3 non-Dummy models** from the leaderboard. 30 trials each.

Objective: maximize primary metric (ROC-AUC for binary, macro F1 for multi-class).

In [ ]:
# Top 3 non-Dummy models from leaderboard (already sorted descending)
top3 = leaderboard[~leaderboard['model'].str.contains('Dummy')].head(3)['model'].tolist()
print(f"Top 3 models selected for Optuna tuning: {top3}")

In [ ]:
def make_objective_clf(model_name, X_tr, y_tr, skf, seed, primary_metric):
    def objective(trial):
        if 'RandomForest' in model_name:
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'max_depth': trial.suggest_int('max_depth', 3, 15),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
            }
            model = RandomForestClassifier(random_state=seed, **params)
        elif 'XGB' in model_name:
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'max_depth': trial.suggest_int('max_depth', 3, 10),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            }
            model = xgb.XGBClassifier(random_state=seed, verbosity=0,
                                       eval_metric='logloss', **params)
        elif 'LGBM' in model_name:
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 300),
                'max_depth': trial.suggest_int('max_depth', 3, 10),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
                'num_leaves': trial.suggest_int('num_leaves', 20, 150),
            }
            model = lgb.LGBMClassifier(random_state=seed, verbose=-1, **params)
        else:
            params = {'C': trial.suggest_float('C', 0.01, 100.0, log=True)}
            model = LogisticRegression(max_iter=1000, random_state=seed, **params)
        
        scores = cross_validate(model, X_tr, y_tr, cv=skf,
                                scoring=primary_metric)
        return scores['test_score'].mean()
    
    return objective

In [ ]:
tuned_rows = []
tuned_models = {}

for model_name in top3:
    print(f"\nTuning {model_name} (30 trials)...")
    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(
        make_objective_clf(model_name, X_train, y_train, skf, SEED, primary_metric),
        n_trials=30, show_progress_bar=False
    )
    
    best_params = study.best_params
    best_score = study.best_value
    print(f"  Best {primary_label}: {best_score:.4f}  |  Params: {best_params}")
    
    # Refit on full training set with best params
    if 'RandomForest' in model_name:
        best_model = RandomForestClassifier(random_state=SEED, **best_params)
    elif 'XGB' in model_name:
        best_model = xgb.XGBClassifier(random_state=SEED, verbosity=0,
                                        eval_metric='logloss', **best_params)
    elif 'LGBM' in model_name:
        best_model = lgb.LGBMClassifier(random_state=SEED, verbose=-1, **best_params)
    else:
        best_model = LogisticRegression(max_iter=1000, random_state=SEED, **best_params)
    
    best_model.fit(X_train, y_train)
    tuned_models[model_name] = best_model
    
    tuned_rows.append({
        'model':            f'{model_name} (tuned)',
        'cv_primary_mean':  round(best_score, 4),
        'cv_primary_std':   None,
        'cv_f1_mean':       None,
        'cv_accuracy_mean': None,
        'test_score':       None,
        'note':             'Optuna-tuned',
    })

tuned_df = pd.DataFrame(tuned_rows)
leaderboard = pd.concat([leaderboard, tuned_df], ignore_index=True).sort_values(
    'cv_primary_mean', ascending=False).reset_index(drop=True)
print(f"\n--- UPDATED LEADERBOARD (with tuned, {primary_label}) ---")
display(leaderboard[['model','cv_primary_mean','note']])

## Step 6: Test Set Reveal ★

The best tuned model is now evaluated on the **held-out test set** (20% of data, sealed since Step 1).

This is the **only time the test set is used**. This score estimates real-world performance.

In [ ]:
# Best tuned model = highest cv_primary_mean among tuned models
best_tuned_name = tuned_df.sort_values('cv_primary_mean', ascending=False).iloc[0]['model']
base_name = best_tuned_name.replace(' (tuned)', '')
best_model = tuned_models[base_name]

y_pred_test = best_model.predict(X_test)

if is_binary:
    y_prob_test = best_model.predict_proba(X_test)[:, 1]
    test_primary = roc_auc_score(y_test, y_prob_test)
    primary_name = 'ROC-AUC'
else:
    test_primary = f1_score(y_test, y_pred_test, average='macro')
    primary_name = 'Macro F1'

test_acc = accuracy_score(y_test, y_pred_test)
test_f1  = f1_score(y_test, y_pred_test, average='weighted')

print(f"Best model: {best_tuned_name}")
print(f"Test {primary_name}: {test_primary:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Weighted F1: {test_f1:.4f}")

# Append test reveal row to leaderboard
test_row = pd.DataFrame([{
    'model':            f'{best_tuned_name} ★ TEST',
    'cv_primary_mean':  test_primary,
    'cv_primary_std':   None,
    'cv_f1_mean':       test_f1,
    'cv_accuracy_mean': test_acc,
    'test_score':       test_primary,
    'note':             'TEST SET — final evaluation',
}])
leaderboard = pd.concat([leaderboard, test_row], ignore_index=True)
display(leaderboard[['model','cv_primary_mean','cv_f1_mean','cv_accuracy_mean','note']])

In [ ]:
# Save best model — MOD-10
model_path = models_dir / 'best_model.pkl'
joblib.dump(best_model, model_path)
print(f"Best model saved: {model_path.relative_to(PROJECT_ROOT)}")

notebook_version = 'v1'  # execute-phase.md sets this; update if running manually

metadata = {
    'model_name':       best_tuned_name,
    'problem_type':     problem_type,
    'cv_metric':        primary_metric,
    'cv_score_mean':    float(tuned_df.sort_values('cv_primary_mean', ascending=False).iloc[0]['cv_primary_mean']),
    'cv_score_std':     None,
    'test_score':       round(float(test_primary), 4),
    'feature_names':    X.columns.tolist(),
    'hyperparameters':  best_model.get_params(),
    'training_date':    str(date.today()),
    'notebook_version': notebook_version,
}

metadata_path = models_dir / 'model_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print(f"Metadata saved: {metadata_path.relative_to(PROJECT_ROOT)}")
print(json.dumps(metadata, indent=2, default=str))

In [ ]:
leaderboard_all_final = pd.read_csv(leaderboard_path) if leaderboard_path.exists() else pd.DataFrame()
leaderboard_all_final = pd.concat(
    [leaderboard_all_final, leaderboard[leaderboard['note'].isin(['Optuna-tuned', 'TEST SET — final evaluation'])]],
    ignore_index=True
)
leaderboard_all_final.to_csv(leaderboard_path, index=False)
print(f"Final leaderboard ({len(leaderboard_all_final)} rows) saved to {leaderboard_path.relative_to(PROJECT_ROOT)}")

## Interpretation & Recommendations

*This section is written by Claude after notebook execution (Decision 8).*

**Instructions for execute-phase.md:** After running this notebook, read `models/leaderboard.csv` and the SHAP output files in `reports/`, then replace this cell with:
1. The top model finding (which model won and by how much vs. baseline)
2. Any anomalies (e.g., low baseline gap suggesting features are weak, or high CV std suggesting overfitting)
3. Suggested next steps

*To create a new modelling iteration with a different focus: run `/doml-iterate-model "your direction here"`*

---

## Caveats & Limitations

- **Correlation is not causation.** Model feature importances reflect statistical association, not causal relationships.
- CV metrics are estimates of generalization error, not guarantees. Performance on new data may differ.
- The test score (Step 6) is the single most reliable estimate of out-of-sample performance — but it is still an estimate on one specific 20% split.
- Optuna tunes hyperparameters on CV score; the optimal hyperparameters for CV may not be optimal for the full training set.
- Class imbalance: if one class is rare, accuracy may be misleading — refer to ROC-AUC and F1 scores.
- Models were not evaluated for fairness or bias across demographic groups. If the target variable or features correlate with protected attributes, additional fairness analysis is required.